# Lab 3 — RAG over resolved ticket summaries


In [1]:
from pathlib import Path

import pandas as pd

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
else:
    for p in [GH_ROOT, *GH_ROOT.parents]:
        if (p / "data" / "support-ops").is_dir():
            GH_ROOT = p
            break
SUPPORT_OPS = GH_ROOT / "data" / "support-ops"

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tickets = pd.read_csv(SUPPORT_OPS / "support_tickets.csv")
resolved = tickets[tickets["status"] == "resolved"].head(50)
docs = resolved["summary"].tolist()
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    emb = model.encode(docs)
    q = model.encode(["BGP peering instability"])
except ImportError:
    vec = TfidfVectorizer()
    emb = vec.fit_transform(docs)
    q = vec.transform(["BGP peering instability"])
idx = cosine_similarity(q, emb).argmax()
print("Closest resolved ticket:", resolved.iloc[idx]["ticket_id"])
print(resolved.iloc[idx]["summary"])


Closest resolved ticket: T00070
Customer reports bgp peering on ASA; intermittent since last change window.
